---
title: "`pytask` Config: Defining the Pipeline Internals in `pytask`"
---

## config

> This is the config module for the `pytask` pipeline. 
This module defines the data catalog(s) and any hard-coded parameters that are used throughout the pipeline.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## `DEV_MODE`: A Quick Development Flag

I'm adding a flag to the config that can be used for quick development. 
If you import this boolean variable, it can be used to skip tasks,
setup samples, etc. on the fly by `marking` a task with the `pytask.mark.skipif`
decorator. Change this to `False` when you're ready to run the full pipeline.

## The Download Task

A good strategy may be to set the hard coded parameters in the config file, and then use the `pytask` data catalog to manage the data. This way, we can easily change the parameters without having to modify the code. This is especially useful for the API query, where we need to be able to set the parameter grid for the years and data types we want to download data for. So, let's create an entry in the data catalog specifically for the download task.

A good strategy I thought about for grid parameter comprehension is to create a dataframe or namedtuple that expands all the combinations of parameters, and then uses each combination to create the tasks which are then easily added to the data catalog. This way, we can still easily inspect the pipeline and see what tasks are being run, while also being able to easily change the parameters in the config file without too much hassle.

An important framework decision I'm making here is that each ROW of the dataframe corresponds to a single task, so that we can quickly understand at a glance what the task is doing, and also easily develop the code for the task itself. This is different from the hydra approach where a job is first specified by a default config, and then the parameters are swept over in the config file. This is a more flexible approach, IMO.

So, to do this, we define one job as a query to the CDS API that must contain:
- The dataset (re-analysis)
- The year
- The month
- All days in the month
- All times of day (hour)
- The geography (region), which will need:
    - The URL to the shapefile to calculate the bounding box

Given one combination of all of these, a single job can complete the first "task" in parallel.

Now, we can use a namedtuple to create a class, `query`, that will be passed directly from the pytask data catalog to the task function.

In [0]:
#| echo: false
#| output: asis
show_doc(Query)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/config.py#L42){target="_blank" style="float:right; font-size:smaller"}

### Query

>      Query (year:str, month:str, day:list[str], time:list[str],
>             geography:dict, product_type:str, variables:list[str])

*A named tuple to hold the query parameters for the download.*

In [ ]:
print(f"Number of estimated jobs: {len(queries)}. Examples...")

for query in queries[:5]:
    print(query)

Number of estimated jobs: 384. Examples...
Query(year=2009, month=1, geography=madagascar)
Query(year=2009, month=1, geography=nepal)
Query(year=2009, month=2, geography=madagascar)
Query(year=2009, month=2, geography=nepal)
Query(year=2009, month=3, geography=madagascar)


Now add them to the catalog. We're going to use a dictionary to
nest data catalogs so that we can return specific task products to
named data catalog nodes.

## The Aggregation Task

To carry out the aggregation, we will follow similar logic to the original pipeline and use xarray to aggregate data into spatial and temporal averages. The aggregation task will take the downloaded data and compute the mean over the specified time period and spatial region. However, in this case, we want to aggregate the data diurnally, so we will need to fetch the sundown and sunrise times for the region and use them to compute the diurnal averages.

To do this, let's use a dataframe as opposed to a list of class objects.